# Tech Challenge Fase 1 — Diagnóstico de Diabetes
## 2. Pré-processamento dos Dados
Este notebook realiza a limpeza e preparação dos dados para modelagem:
- Tratamento de zeros suspeitos
- Escalonamento de features
- Separação treino/validação/teste

## 2.1 Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os

sns.set_theme(style='whitegrid')
print('Bibliotecas importadas com sucesso!')

## 2.2 Carregamento do dataset

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
print(f'Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas')
df.head()

## 2.3 Tratamento de zeros suspeitos

Colunas onde zero é fisiologicamente impossível ou improvável serão tratadas:
substituímos 0 por NaN e depois imputamos pela **mediana** (mais robusta a outliers que a média).

In [ ]:
df_clean = df.copy()

colunas_suspeitas = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Substituir zeros por NaN
df_clean[colunas_suspeitas] = df_clean[colunas_suspeitas].replace(0, np.nan)

print('=== VALORES NULOS APÓS SUBSTITUIÇÃO DE ZEROS ===')
print(df_clean.isnull().sum())

In [ ]:
# Imputar pela mediana de cada coluna
for col in colunas_suspeitas:
    mediana = df_clean[col].median()
    df_clean[col].fillna(mediana, inplace=True)
    print(f'{col}: mediana usada = {mediana:.2f}')

print('\n=== VALORES NULOS APÓS IMPUTAÇÃO ===')
print(df_clean.isnull().sum())
print('\nNenhum valor nulo restante!' if df_clean.isnull().sum().sum() == 0 else 'Ainda há valores nulos!')

## 2.4 Comparação antes x depois do tratamento

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, col in enumerate(colunas_suspeitas):
    # Antes
    axes[0, i].hist(df[col], bins=30, color='tomato', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col}\n(Antes)', fontsize=11)
    axes[0, i].set_ylabel('Frequência')

    # Depois
    axes[1, i].hist(df_clean[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'{col}\n(Depois)', fontsize=11)
    axes[1, i].set_ylabel('Frequência')

plt.suptitle('Comparação: Antes x Depois do Tratamento de Zeros', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/antes_depois_tratamento.png', dpi=150)
plt.show()

## 2.5 Separação de features e variável alvo

In [ ]:
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

print(f'Features (X): {X.shape}')
print(f'Alvo (y): {y.shape}')
print(f'\nFeatures usadas: {list(X.columns)}')

## 2.6 Separação treino / validação / teste

Estratégia: **70% treino | 15% validação | 15% teste**

Usamos `stratify=y` para manter a proporção de classes em cada conjunto (importante em datasets desbalanceados).

In [ ]:
# Primeiro split: 70% treino, 30% temporário
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Segundo split: 50% do temporário = validação, 50% = teste (15% cada do total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('=== TAMANHO DOS CONJUNTOS ===')
print(f'Treino:    {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Teste:     {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')

print('\n=== PROPORÇÃO DE CLASSES ===')
for nome, y_set in [('Treino', y_train), ('Validação', y_val), ('Teste', y_test)]:
    pct = y_set.value_counts(normalize=True) * 100
    print(f'{nome}: Sem diabetes={pct[0]:.1f}% | Com diabetes={pct[1]:.1f}%')

## 2.7 Escalonamento das features (StandardScaler)

O scaler é **treinado apenas no conjunto de treino** para evitar data leakage.
Depois aplicamos a mesma transformação em validação e teste.

In [ ]:
scaler = StandardScaler()

# Fit apenas no treino!
X_train_scaled = scaler.fit_transform(X_train)

# Transform nos demais
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Converter de volta para DataFrame (facilita visualização)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('=== ESTATÍSTICAS APÓS ESCALONAMENTO (treino) ===')
print(X_train_scaled.describe().round(2))

## 2.8 Salvar dados processados

In [ ]:
# Criar pasta processed dentro de data/
os.makedirs('../data/processed', exist_ok=True)

# Salvar conjuntos
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_val_scaled.to_csv('../data/processed/X_val.csv',     index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv',        index=False)
y_val.to_csv('../data/processed/y_val.csv',            index=False)
y_test.to_csv('../data/processed/y_test.csv',          index=False)

# Salvar o scaler para reutilizar no main.py
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Dados processados salvos em data/processed/')
print(os.listdir('../data/processed/'))

## 2.9 Conclusões do Pré-processamento

**O que foi feito:**

1. **Zeros suspeitos tratados:** colunas Glucose, BloodPressure, SkinThickness, Insulin e BMI tiveram zeros substituídos por NaN e imputados pela mediana
2. **Separação estratificada:** 70% treino / 15% validação / 15% teste, mantendo proporção de classes
3. **Escalonamento:** StandardScaler aplicado com fit apenas no treino (evita data leakage)
4. **Dados salvos:** conjuntos prontos em `data/processed/` para uso na modelagem

**Próximo passo:** treinar e comparar modelos de classificação (Regressão Logística e Random Forest)